In [0]:
# Databricks notebook source
# GOLD LAYER: Constructor/Team Championship Standings
# Purpose: Aggregate both drivers' points per team per race
# Source: f1_data.silver.fact_results

from pyspark.sql.functions import (
    col, sum as spark_sum, count, when, lit,
    round as spark_round, dense_rank, collect_list,
    concat_ws, countDistinct, current_timestamp,
    max as spark_max, min as spark_min, avg, row_number
)
from pyspark.sql.window import Window

# ============ 1. READ FROM SILVER ============
fact = spark.read.table("f1_data.silver.fact_results")

race_results = fact.filter(col("session_type").isin("Race", "Sprint"))

# ============ 2. CUMULATIVE TEAM POINTS ============
# Define a window that parititions the data by team name and year and orders by session date
team_cumulative_window = Window.partitionBy("team_name", "year") \
    .orderBy("session_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Define a new table that aggregates per team per race
team_per_race = race_results.groupBy(
    "team_name", "year", "meeting_key", "meeting_name",
    "race_country", "race_location", "session_date", "session_key"
).agg(
    spark_sum("points").alias("race_points"), # Take the sum of points from both racers
    collect_list("points").alias("individual_points_breakdown"), # Display the original points of both racers
    count("driver_number").alias("drivers_fielded"), # Count the number of drivers fielded
    spark_sum(when(col("position") == 1, 1).otherwise(0)).alias("race_wins"), # Count the number of wins when position says 1
    spark_sum(when(col("position") <= 3, 1).otherwise(0)).alias("race_podiums"), # Count the number of podiums when position is in top 3
    spark_sum(when(col("dnf") == True, 1).otherwise(0)).alias("race_dnfs"), # Count the instances of dnf
    spark_min("position").alias("best_driver_position"), # Determine the best finishing position of the team 
    collect_list("driver_name").alias("driver_names") # Display both drivers' names
)

# Cumulative team stats over season
team_cumulative = team_per_race \
    .withColumn("cumulative_team_points",
        spark_sum("race_points").over(team_cumulative_window)
    ) \
    .withColumn("cumulative_wins",
        spark_sum("race_wins").over(team_cumulative_window)
    ) \
    .withColumn("cumulative_podiums",
        spark_sum("race_podiums").over(team_cumulative_window)
    ) \
    .withColumn("cumulative_dnfs",
        spark_sum("race_dnfs").over(team_cumulative_window)
    ) \
    .withColumn("cumulative_races",
        count("session_key").over(team_cumulative_window)
    )

# ============ 3. FINAL TEAM STANDINGS ============
# Define a window that partitions the data by team name and year and orders by session date in descending order
latest_team_window = Window.partitionBy("team_name", "year") \
    .orderBy(col("session_date").desc())

# Define a table where the row numbers are each team's entry, and since it is in descending, the latest entry will be the first entry
# Keep only the latest entry
team_latest = team_cumulative \
    .withColumn("row_num", row_number().over(latest_team_window)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

# Define a window that partitions by year and orderss by team points in descending order
# This would make it that the first instance of the year would contain the highest ponits
constructor_window = Window.partitionBy("year") \
    .orderBy(col("cumulative_team_points").desc())

# DENSE_RANK provides consecutive ranks without any gaps while rank skips numbers after a tie
team_standings = team_latest \
    .withColumn("constructor_position", # Define a column that implements ranks for constructors based on their points earned across the year
        dense_rank().over(constructor_window)
    ) \
    .withColumn("points_behind_leader", # Define a column that displays the deficit a constructor has against the lead constructor
        spark_max("cumulative_team_points").over(Window.partitionBy("year"))
        - col("cumulative_team_points")
    ) \
    .withColumn("avg_points_per_race", # Avg points = total points / total races
        spark_round(col("cumulative_team_points") / col("cumulative_races"), 2)
    ) \
    .withColumn("dnf_rate", # DNF rate = total dnf / total races
        spark_round(col("cumulative_dnfs") / (col("cumulative_races") * 2) * 100, 2)
    ) \
    .withColumn("gold_processed_at", current_timestamp()) \
    .withColumn("gold_source", lit("f1_data.silver.fact_results"))

# ============ 4. WRITE TO GOLD ============
team_standings.select(
    "team_name",
    "year",
    "constructor_position",
    col("cumulative_team_points").alias("total_points"),
    "points_behind_leader",
    col("cumulative_races").alias("races_contested"),
    col("cumulative_wins").alias("total_wins"),
    col("cumulative_podiums").alias("total_podiums"),
    col("cumulative_dnfs").alias("total_dnfs"),
    "avg_points_per_race",
    "dnf_rate",
    col("session_date").alias("last_race_date"),
    col("meeting_name").alias("last_race_name"),
    "gold_processed_at",
    "gold_source"
).write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("year") \
    .saveAsTable("f1_data.gold.team_standings")

print(f"✓ gold.team_standings written")

print("\n🏆 CONSTRUCTOR STANDINGS:")
spark.sql("""
    SELECT
        constructor_position,
        team_name,
        total_points,
        total_wins,
        total_podiums,
        avg_points_per_race,
        dnf_rate
    FROM f1_data.gold.team_standings
    WHERE year = 2024
    ORDER BY constructor_position
""").show(truncate=False)

✓ gold.team_standings written

🏆 CONSTRUCTOR STANDINGS:
+--------------------+---------------+------------+----------+-------------+-------------------+--------+
|constructor_position|team_name      |total_points|total_wins|total_podiums|avg_points_per_race|dnf_rate|
+--------------------+---------------+------------+----------+-------------+-------------------+--------+
|1                   |McLaren        |516.0       |5         |20           |24.57              |2.38    |
|2                   |Red Bull Racing|475.0       |10        |20           |22.62              |9.52    |
|3                   |Ferrari        |441.0       |3         |15           |21.0               |7.14    |
|4                   |Mercedes       |329.0       |3         |8            |15.67              |7.14    |
|5                   |Aston Martin   |86.0        |0         |0            |4.1                |9.52    |
|6                   |RB             |34.0        |0         |0            |1.62               |

In [0]:
display(team_cumulative)

team_name,year,meeting_key,meeting_name,race_country,race_location,session_date,session_key,race_points,individual_points_breakdown,drivers_fielded,race_wins,race_podiums,race_dnfs,best_driver_position,driver_names,cumulative_team_points,cumulative_wins,cumulative_podiums,cumulative_dnfs,cumulative_races
Alpine,2024,1229,Bahrain Grand Prix,Bahrain,Sakhir,2024-03-02T15:00:00.000Z,9472,0.0,"List(0.0, 0.0)",2,0,0,0,17.0,"List(Esteban Ocon, Pierre Gasly)",0.0,0,0,0,1
Alpine,2024,1230,Saudi Arabian Grand Prix,Saudi Arabia,Jeddah,2024-03-09T17:00:00.000Z,9480,0.0,"List(0.0, 0.0)",2,0,0,1,13.0,"List(Esteban Ocon, Pierre Gasly)",0.0,0,0,1,2
Alpine,2024,1231,Australian Grand Prix,Australia,Melbourne,2024-03-24T04:00:00.000Z,9488,0.0,"List(0.0, 0.0)",2,0,0,0,13.0,"List(Pierre Gasly, Esteban Ocon)",0.0,0,0,1,3
Alpine,2024,1232,Japanese Grand Prix,Japan,Suzuka,2024-04-07T05:00:00.000Z,9496,0.0,"List(0.0, 0.0)",2,0,0,0,15.0,"List(Esteban Ocon, Pierre Gasly)",0.0,0,0,1,4
Alpine,2024,1233,Chinese Grand Prix,China,Shanghai,2024-04-20T03:00:00.000Z,9672,0.0,"List(0.0, 0.0)",2,0,0,0,13.0,"List(Esteban Ocon, Pierre Gasly)",0.0,0,0,1,5
Alpine,2024,1233,Chinese Grand Prix,China,Shanghai,2024-04-21T07:00:00.000Z,9673,0.0,"List(0.0, 0.0)",2,0,0,0,11.0,"List(Esteban Ocon, Pierre Gasly)",0.0,0,0,1,6
Alpine,2024,1234,Miami Grand Prix,United States,Miami,2024-05-04T16:00:00.000Z,9506,0.0,"List(0.0, 0.0)",2,0,0,0,9.0,"List(Pierre Gasly, Esteban Ocon)",0.0,0,0,1,7
Alpine,2024,1234,Miami Grand Prix,United States,Miami,2024-05-05T20:00:00.000Z,9507,1.0,"List(1.0, 0.0)",2,0,0,0,10.0,"List(Esteban Ocon, Pierre Gasly)",1.0,0,0,1,8
Alpine,2024,1235,Emilia Romagna Grand Prix,Italy,Imola,2024-05-19T13:00:00.000Z,9515,0.0,"List(0.0, 0.0)",2,0,0,0,14.0,"List(Esteban Ocon, Pierre Gasly)",1.0,0,0,1,9
Alpine,2024,1236,Monaco Grand Prix,Monaco,Monaco,2024-05-26T13:00:00.000Z,9523,1.0,"List(1.0, 0.0)",2,0,0,1,10.0,"List(Pierre Gasly, Esteban Ocon)",2.0,0,0,2,10
